In [3]:
# Cell 1: Install core dependencies
%pip install python-dotenv langchain langchain-openai trafilatura -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Cell 2: Load environment variables and verify keys are present
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

assert OPENAI_API_KEY, "OPENAI_API_KEY not found in .env"
assert SERPER_API_KEY, "SERPER_API_KEY not found in .env"

print(f"OpenAI model : {OPENAI_MODEL}")
print(f"OpenAI key   : ...{OPENAI_API_KEY[-6:]}")
print(f"Serper key   : ...{SERPER_API_KEY[-6:]}")

OpenAI model : gpt-4o-mini
OpenAI key   : ...i-lLAA
Serper key   : ...d053d8


In [5]:
# Cell 2b: Disk-based cache for Serper and LLM calls
# Avoids re-calling paid APIs when re-running notebook cells.
# Cache lives in .cache/ as JSON files keyed by SHA-256 of the input.
import hashlib, json, pathlib

CACHE_DIR = pathlib.Path(".cache")
CACHE_DIR.mkdir(exist_ok=True)

def _cache_key(*parts) -> str:
    """Create a deterministic hash from arbitrary string parts."""
    raw = json.dumps(parts, sort_keys=True, ensure_ascii=True)
    return hashlib.sha256(raw.encode()).hexdigest()

def cache_get(namespace: str, *key_parts):
    """Return cached value or None if miss."""
    h = _cache_key(*key_parts)
    p = CACHE_DIR / namespace / f"{h}.json"
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    return None

def cache_set(namespace: str, value, *key_parts):
    """Write value to cache."""
    h = _cache_key(*key_parts)
    d = CACHE_DIR / namespace
    d.mkdir(exist_ok=True)
    p = d / f"{h}.json"
    p.write_text(json.dumps(value, ensure_ascii=False, indent=1), encoding="utf-8")

def cache_stats():
    """Print cache stats per namespace."""
    if not CACHE_DIR.exists():
        print("Cache: empty")
        return
    for ns in sorted(CACHE_DIR.iterdir()):
        if ns.is_dir():
            files = list(ns.glob("*.json"))
            print(f"  {ns.name}: {len(files)} entries")

print(f"Cache dir: {CACHE_DIR.resolve()}")
cache_stats()

Cache dir: C:\Users\Abhishek A\Defining_Category\.cache
  llm_scoring: 16 entries
  llm_synthesis: 1 entries
  serper: 64 entries


In [ ]:
# Cell 3: Category configuration and tier-based search setup
# Complete pipeline: Category → Tiered Search → Blocklist → Scrape → Filter → Score → Synthesize

# ── Category definition ───────────────────────────────────────
TEST_CATEGORY = "Account-Based Marketing"
CATEGORY_MATURITY = "evolving"  # Determines age thresholds

# All aliases for comprehensive search
CATEGORY_ALIASES = [
    "Account-Based Marketing",
    "ABM",
    "Account-Based Marketing Platforms", 
    "ABM platforms",
    "Account-Based Everything",
    "ABX",
    "Account-Based Experience",
]

# ── Tier-based site prioritization (ascending order: Tier1 → Tier2 → Tier3) ─────────────
# Tier 1: Major industry analysts (highest priority)
TIER1_SITES = [
    "gartner.com",                    # Root domain for all gartner.com URLs
    "blogs.gartner.com",
    "gartner.com/en/articles",
    "gartner.com/en/marketing/glossary",
    "gartner.com/en/information-technology/glossary",
    "gartner.com/en/sales/glossary",
    "forrester.com",
    "go.forrester.com",
    "idc.com",
    "blogs.idc.com",
]

# Tier 2: Independent analysts (second priority)
TIER2_SITES = [
    "constellationr.com",
    "infotech.com",
    "451research.com",
    "spglobal.com/marketintelligence",
    "omdia.tech.informa.com",
    "hfsresearch.com",
    "isg-one.com",
    "everestgrp.com",
    "nucleusresearch.com",
    "dresneradvisory.com",
    "abiresearch.com",
    "gigaom.com",
    "aragonresearch.com",
    "moorinsightsstrategy.com",
    "pund-it.com",
    "enderlegroup.com",
    "jgoldassociates.com",
]

# Tier 2b: Domain-specialist analysts
TIER2B_SITES = [
    "kuppingercole.com",
    "barc.com",
    "frost.com",
    "enterprisemanagement.com",
    "esg-global.com",
    "tag-cyber.com",
    "securosis.com",
    "colemanparkes.com",
]

# Tier 3: Trade publications (lowest priority)
TIER3_SITES = [
    "chiefmartec.com",
    "martech.org",
    "adexchanger.com",
    "digiday.com",
    "searchcrm.com",
    "searchsecurity.com",
    "searchdatamanagement.com",
    "cio.com",
    "computerworld.com",
    "infoworld.com",
    "csoonline.com",
    "darkreading.com",
    "informationweek.com",
    "diginomica.com",
    "theregister.com",
    "zdnet.com",
    "stratechery.com",
    "tomtunguz.com",
    "ben-evans.com",
]

# Tier 4: Practitioner/consultancy publications
CONSULTANCY_SITES = [
    "mckinsey.com",
    "bcg.com",
    "bain.com",
    "deloitte.com/insights",
    "accenture.com",
    "ey.com",
    "kpmg.com",
    "a16z.com",
]

# Tier 5: Academic / standards
ACADEMIC_SITES = [
    "hbr.org",
    "sloanreview.mit.edu",
    "nist.gov",
]

# ── Blocklist: URLs and sub-paths to delete/ignore ─────────────────────────────────────
# These will be filtered out during search and post-processing
BLOCKLIST = {
    "url_patterns": [
        "/software-reviews/",       # Info-Tech SoftwareReviews = review platform
        "/compare/",                # head-to-head comparison pages
        "/products/",               # product review pages
        "gpivendorresources",       # Gartner vendor portal
        "gartner.com/reviews",      # Gartner Peer Insights
        "gartner.com/en/digital-markets",  # Capterra/GetApp/Software Advice
        "g2.com", "trustradius.com", "capterra.com", "getapp.com",
        "sourceforge.net", "goodfirms.co", "crozdesk.com",
        "/sponsors/",               # Event/sponsor pages
        "/event/",
        "/wp-content/uploads/",     # WordPress uploaded PDFs/images
        "/content/dam/",            # CMS asset dirs (Accenture, Deloitte)
        "/docs/default-source/",    # ISG document library
        "/downloads/",              # IDC/misc download dirs
        "event-pdf-generator",      # Forrester event PDFs
    ],
    "domains": [
        "store.frost.com",          # Frost paywall store
        "my.idc.com",               # IDC paywalled docs
        "info.idc.com",             # IDC gated lead-gen content
        "view.frost.com",           # Frost gated viewer
        "hub.frost.com",            # Frost hub pages
        "web-assets.bcg.com",       # BCG PDF/asset CDN
        "keithdawson.isg-one.com",  # personal blog subdomain
        "portal.gigaom.com",       # GigaOm paywalled portal
        # Vendor sites to block
        "optimizely.com",
        "salesforce.com",
        "linkedin.com",
        "wikipedia.org",
        "adobe.com",
        "oracle.com",
        "demandbase.com",
        "cognism.com",
        "zoomforth.com",
        "factors.ai",
        "influ2.com",
        "hginsights.com",
        "mutinyhq.com",
        "marketone.com",
        "strategicabm.com",
        "clay.com",
        "youtube.com",
    ]
}

# ── Analyst hub URLs (direct crawl targets) ───────────────────────────────────────
ANALYST_HUB_URLS = [
    "https://www.forrester.com/blogs/author/john_arnold/",
    "https://www.forrester.com/blogs/author/jessie_johnson/",
    "https://www.forrester.com/blogs/author/terry_flaherty/",
    "https://www.gartner.com/en/articles/the-account-based-everything-framework",
    "https://research.isg-one.com/analyst-perspectives/topic/intelligent-marketing",
]

# ── Configuration ───────────────────────────────────────
ALL_TRUSTED_SITES = (TIER1_SITES + TIER2_SITES + TIER2B_SITES + TIER3_SITES + 
                     CONSULTANCY_SITES + ACADEMIC_SITES)

# Currency thresholds based on maturity
CURRENCY_THRESHOLDS = {"emerging": 12, "evolving": 24, "stable": 36}
MAX_SOURCE_AGE_MONTHS = CURRENCY_THRESHOLDS[CATEGORY_MATURITY]

# Build exclusion string for Serper queries
_exc_parts = ["-filetype:pdf"]
_exc_parts += [f"-site:{s}" for s in BLOCKLIST["domains"]]
_exc_parts += [f'-inurl:"{p}"' for p in BLOCKLIST["url_patterns"]]
SERPER_EXCLUSIONS = " ".join(_exc_parts)

print(f"Category: {TEST_CATEGORY}")
print(f"Maturity: {CATEGORY_MATURITY} (max age: {MAX_SOURCE_AGE_MONTHS} months)")
print(f"Aliases: {len(CATEGORY_ALIASES)}")
print(f"Tiers: T1={len(TIER1_SITES)}, T2={len(TIER2_SITES)}, T3={len(TIER3_SITES)}")
print(f"Blocklist: {len(BLOCKLIST['domains'])} domains, {len(BLOCKLIST['url_patterns'])} patterns")
print(f"Total trusted sites: {len(ALL_TRUSTED_SITES)}")

Category      : Account-Based Marketing
Maturity      : evolving (max source age: 24 months)
Aliases       : 7
Tier 1 sites  : 10 (Gartner, Forrester, IDC)
Tier 2 sites  : 17 (independent analysts)
Tier 2b sites : 8 (domain specialists)
Trade pubs    : 19
Consultancies : 8
Academic      : 3
Total sites   : 65
Analyst hubs  : 5
Drop patterns : 15
Serper excl.  : 8 sites, 5 inurl, 17 vendor sites, +pdf filter

Exclusion string (648 chars):
  -filetype:pdf -site:store.frost.com -site:my.idc.com -site:info.idc.com -site:view.frost.com -site:hub.frost.com -site:web-assets.bcg.com -site:keithdawson.isg-one.com -site:portal.gigaom.com -inurl:"/wp-content/uploads/" -inurl:"/content/dam/" -inurl:"/docs/default-source/" -inurl:"/downloads/" -inurl:"event-pdf-generator" -site:optimizely.com -site:salesforce.com -site:linkedin.com -site:wikipedia.org -site:adobe.com -site:oracle.com -site:demandbase.com -site:cognism.com -site:zoomforth.com -site:factors.ai -site:influ2.com -site:hginsights.com -s

In [ ]:
# Cell 4: Tier-based site-restricted search using Serper
# Prioritizes Tier1 → Tier2 → Tier3 in ascending order
import requests, time
from urllib.parse import urlparse

SEARCH_DELAY = 0.3  # Rate limiting
RESULTS_PER_QUERY = 10  # Number of results per query

def serper_search(query: str, api_key: str, num: int = 10) -> list[dict]:
    """Call Serper.dev Google Search API with caching."""
    cached = cache_get("serper", query, num, {})
    if cached is not None:
        return cached
    
    payload = {"q": query, "num": num}
    resp = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": api_key, "Content-Type": "application/json"},
        json=payload,
        timeout=15,
    )
    resp.raise_for_status()
    results = resp.json().get("organic", [])
    cache_set("serper", results, query, num, {})
    time.sleep(SEARCH_DELAY)
    return results

def batch_sites(sites: list[str], batch_size: int = 5) -> list[str]:
    """Batch sites for OR queries."""
    batches = []
    for i in range(0, len(sites), batch_size):
        chunk = sites[i : i + batch_size]
        clause = " OR ".join(f"site:{s}" for s in chunk)
        batches.append(f"({clause})")
    return batches

def is_blocked(url: str) -> bool:
    """Check if URL matches blocklist patterns."""
    url_lower = url.lower()
    hostname = urlparse(url).netloc.lower()
    
    # Check domain blocklist
    if any(blocked in hostname for blocked in BLOCKLIST["domains"]):
        return True
    
    # Check URL pattern blocklist
    for pattern in BLOCKLIST["url_patterns"]:
        if pattern in url_lower:
            return True
    
    return False

def search_tier(tier_name: str, sites: list[str], aliases: list[str], 
                seen: set, results: list, num_per_query: int = RESULTS_PER_QUERY):
    """Search a specific tier with all aliases."""
    batches = batch_sites(sites, batch_size=3)
    queries_run = 0
    hits_added = 0
    blocked = 0
    cache_hits = 0
    
    print(f"\n── Searching {tier_name} ──")
    
    for alias in aliases:
        for site_batch in batches:
            query = f'{site_batch} "{alias}" {SERPER_EXCLUSIONS}'
            queries_run += 1
            
            try:
                was_cached = cache_get("serper", query, num_per_query, {}) is not None
                if was_cached:
                    cache_hits += 1
                    
                hits = serper_search(query, SERPER_API_KEY, num=num_per_query)
                
                for hit in hits:
                    url = hit.get("link", "")
                    if not url or url in seen:
                        continue
                    if is_blocked(url):
                        blocked += 1
                        continue
                    
                    seen.add(url)
                    results.append({
                        "url": url,
                        "title": hit.get("title", ""),
                        "snippet": hit.get("snippet", ""),
                        "query_alias": alias,
                        "search_pass": tier_name,
                    })
                    hits_added += 1
                    
            except Exception as e:
                print(f"    ✗ Query failed: {e}")
    
    cached_msg = f", {cache_hits} from cache" if cache_hits else ""
    print(f"  {tier_name}: {queries_run} queries → {hits_added} URLs (blocked {blocked}{cached_msg})")
    return queries_run

# ── Execute tier-based search ───────────────────────────────────────────────────────
seen_urls = set()
all_results = []
total_queries = 0

print("=" * 70)
print(f"TIER-BASED SEARCH: {TEST_CATEGORY}")
print(f"Priority: Tier1 → Tier2 → Tier3")
print("=" * 70)

# Add analyst hub URLs first (highest priority)
print("\n── Analyst Hub URLs ──")
hub_added = 0
for hub_url in ANALYST_HUB_URLS:
    if hub_url not in seen_urls and not is_blocked(hub_url):
        seen_urls.add(hub_url)
        all_results.append({
            "url": hub_url,
            "title": f"[Hub] {hub_url.split('/')[-2] if hub_url.endswith('/') else hub_url.split('/')[-1]}",
            "snippet": "",
            "query_alias": TEST_CATEGORY,
            "search_pass": "AnalystHub",
        })
        hub_added += 1
print(f"  AnalystHub: {hub_added} direct URLs")

# Search Tier 1 (highest priority)
total_queries += search_tier("Tier1", TIER1_SITES, CATEGORY_ALIASES, seen_urls, all_results)

# Search Tier 2 (second priority)
total_queries += search_tier("Tier2", TIER2_SITES + TIER2B_SITES, CATEGORY_ALIASES, seen_urls, all_results)

# Search Tier 3 (lowest priority)
total_queries += search_tier("Tier3", TIER3_SITES, CATEGORY_ALIASES, seen_urls, all_results)

# ── Post-search filtering and summary ───────────────────────────────────────────────
print(f"\n{'=' * 70}")
print(f"SEARCH COMPLETE: {total_queries} total queries → {len(all_results)} unique URLs")
print(f"{'=' * 70}")

# Show tier breakdown
from collections import Counter
tier_counts = Counter(r["search_pass"] for r in all_results)
for tier, count in tier_counts.items():
    print(f"  {tier}: {count} URLs")

print(f"\nAll URLs found:")
for i, r in enumerate(all_results):
    print(f"  {i+1}. [{r['search_pass']}] {r['title'][:80]}")
    print(f"     {r['url']}")

cache_stats()

SEARCH: Account-Based Marketing  (Tier 1 + key Tier 2 only)
Primary aliases: ['Account-Based Marketing', 'ABM platforms']
Secondary aliases (Tier 1 only): ['Account-Based Marketing Platforms', 'Account-Based Everything']

── Pass 0: Analyst author-hub URLs ──
  AnalystHub: 5 direct URLs added

── Pass 1: Tier 1 (Gartner, Forrester, IDC) — primary aliases ──
  Tier1-primary: 8 queries → 18 new URLs (blocked 5, 8 from cache)

── Pass 1b: Tier 1 — secondary aliases ──
  Tier1-secondary: 8 queries → 10 new URLs (blocked 2, 6 from cache)

── Pass 2: Key Tier 2 (Constellation, ISG, GigaOm, Nucleus, Aragon) ──
  Tier2-key: 2 queries → 6 new URLs (blocked 0, 2 from cache)

── Post-search filter ──
  Kept 39 URLs from trusted domains
  Dropped 0 URLs from untrusted/vendor domains

TOTAL: 18 Serper queries → 39 unique URLs (post-filter)
  AnalystHub: 5 URLs
  Tier1-primary: 18 URLs
  Tier1-secondary: 10 URLs
  Tier2-key: 6 URLs

Quality breakdown:
  Tier 1: 28 URLs (71.8%)
  Tier 2: 6 URLs (15.4

In [ ]:
# Cell 5: Scrape sites and extract metadata
# Gets Author name, date, content from all discovered URLs
import trafilatura
import time as _time
from datetime import datetime, timedelta

SCRAPE_DELAY = 0.5  # Be polite to servers

def extract_article(url: str) -> dict | None:
    """Download and extract article content with metadata."""
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return {"_error": "fetch returned empty (403/timeout/JS-only)"}
        
        text = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=False,
            output_format="txt",
        )
        if not text:
            return {"_error": "extraction returned no text (template page?)"}
        
        meta = trafilatura.metadata.extract_metadata(downloaded)
        return {
            "url": url,
            "title": meta.title if meta else "",
            "author": meta.author if meta else "",
            "date": meta.date if meta else "",
            "text": text,
            "hostname": meta.sitename if meta else "",
        }
    except Exception as e:
        return {"_error": f"exception: {type(e).__name__}: {e}"}

def source_age_months(date_str: str) -> int | None:
    """Calculate source age in months from date string."""
    if not date_str:
        return None
    try:
        dt = datetime.strptime(date_str[:10], "%Y-%m-%d")
        delta = datetime.now() - dt
        return int(delta.days / 30.44)
    except (ValueError, TypeError):
        return None

# ── Scrape all URLs ───────────────────────────────────────────────────────────────
scraped_sources = []
scrape_failures = []

print(f"Scraping {len(all_results)} URLs...")
print("=" * 60)

for i, result in enumerate(all_results):
    print(f"[{i+1}/{len(all_results)}] {result['url'][:80]}…", end=" ")
    
    article = extract_article(result["url"])
    if article and "_error" not in article and article["text"] and len(article["text"]) > 100:
        # Add search metadata
        article["query_alias"] = result["query_alias"]
        article["search_pass"] = result["search_pass"]
        
        # Check age
        age = source_age_months(article.get("date"))
        if age is not None and age > MAX_SOURCE_AGE_MONTHS:
            scrape_failures.append({
                "url": result["url"], 
                "reason": f"too old ({age} months, threshold {MAX_SOURCE_AGE_MONTHS})"
            })
            print(f"✗ (too old: {age}mo)")
        else:
            scraped_sources.append(article)
            print(f"✓ ({len(article['text'])} chars, age: {age}mo if age else 'unknown'}}")
    else:
        reason = article.get("_error", "too short") if article else "returned None"
        scrape_failures.append({"url": result["url"], "reason": reason})
        print(f"✗ ({reason})")
    
    _time.sleep(SCRAPE_DELAY)

print(f"\n{'=' * 60}")
print(f"SCRAPE RESULTS:")
print(f"  Successfully scraped: {len(scraped_sources)}")
print(f"  Failures: {len(scrape_failures)}")
print(f"  Success rate: {len(scraped_sources)/(len(all_results))*100:.1f}%")

if scrape_failures:
    print(f"\nFailed URLs (first 10):")
    for f in scrape_failures[:10]:
        print(f"  ✗ {f['url'][:80]}… — {f['reason']}")

# Quality breakdown by tier
print(f"\nScraped sources by tier:")
tier_scrapes = {}
for s in scraped_sources:
    tier = s.get("search_pass", "Unknown")
    tier_scrapes[tier] = tier_scrapes.get(tier, 0) + 1

for tier, count in tier_scrapes.items():
    pct = count / len(scraped_sources) * 100 if scraped_sources else 0
    print(f"  {tier}: {count} sources ({pct:.1f}%)")

[1/39] https://research.isg-one.com/analyst-perspectives/topic/intelligent-marketing… ✓ (408 chars)
[2/39] https://www.forrester.com/blogs/author/jessie_johnson/… ✓ (7573 chars)
[3/39] https://www.forrester.com/blogs/author/john_arnold/… ✗ (fetch returned empty (403/timeout/JS-only))
[4/39] https://www.forrester.com/blogs/author/terry_flaherty/… ✓ (5853 chars)
[5/39] https://www.gartner.com/en/articles/the-account-based-everything-framework… ✗ (fetch returned empty (403/timeout/JS-only))
[6/39] https://www.gartner.com/en/documents/6479639… ✗ (fetch returned empty (403/timeout/JS-only))
[7/39] https://www.idc.com/resource-center/blog/gain-share-with-account-based-marketing… ✓ (5640 chars)
[8/39] https://www.forrester.com/report/account-based-everything-what-happens-when-inte… ✓ (702 chars)
[9/39] https://www.forrester.com/blogs/category/account-based-marketing-abm/… ✓ (6132 chars)
[10/39] https://www.forrester.com/blogs/what-is-account-based-marketing/… ✓ (7884 chars)
[11/39] https://ww

In [9]:
# Cell 6: Preview scraped sources (first 5)
print(f"Previewing first {min(5, len(scraped_sources))} scraped sources:\n")
for i, s in enumerate(scraped_sources[:5]):
    print(f"--- Source {i+1} ---")
    print(f"  Title  : {s['title']}")
    print(f"  Author : {s['author']}")
    print(f"  Date   : {s['date']}")
    print(f"  Host   : {s['hostname']}")
    print(f"  Alias  : {s['query_alias']}")
    print(f"  Length  : {len(s['text'])} chars")
    print(f"  Preview: {s['text'][:200]}…")
    print()

Previewing first 5 scraped sources:

--- Source 1 ---
  Title  : ISG Software Research Analyst Perspectives | intelligent marketing
  Author : Keith Dawson
  Date   : 2024-02-05
  Host   : Ventanaresearch
  Alias  : Account-Based Marketing
  Length  : 408 chars
  Preview: Ventana Research recently announced its 2024 Market Agenda in the expertise area of Marketing, continuing the guidance we have offered for nearly two decades to help enterprises derive optimal value f…

--- Source 2 ---
  Title  : Jessie Johnson
  Author : Jessie Johnson
  Date   : 2026-03-10
  Host   : Forrester
  Alias  : Account-Based Marketing
  Length  : 7573 chars
  Preview: Jessie Johnson
Principal Analyst
Author Insights
Blog
The Future Of B2B GTM Isn’t Human Versus AI
AI has long been embedded in the B2B tech stack and go-to-market workflows. The sudden ubiquity of gen…

--- Source 3 ---
  Title  : Terry Flaherty
  Author : Terry Flaherty
  Date   : 2026-04-02
  Host   : Forrester
  Alias  : Account-Based Mar

In [ ]:
# Cell 7: Content deduplication
# Remove near-identical scraped content based on content hashing
import hashlib as _hl

def _content_hash(text: str) -> str:
    """Generate hash from first 500 chars for deduplication."""
    return _hl.md5(text[:500].strip().lower().encode()).hexdigest()

_seen_hashes = set()
deduped_sources = []
duplicates_removed = 0

print("Deduplicating content...")
print("=" * 60)

for s in scraped_sources:
    h = _content_hash(s["text"])
    if h in _seen_hashes:
        duplicates_removed += 1
        continue
    _seen_hashes.add(h)
    deduped_sources.append(s)

print(f"Before dedup: {len(scraped_sources)} sources")
print(f"Duplicates removed: {duplicates_removed}")
print(f"After dedup: {len(deduped_sources)} unique sources")

# Replace scraped_sources with deduped version
scraped_sources = deduped_sources

Before dedup: 28 sources
Duplicates removed: 0
After dedup: 28 unique sources


In [ ]:
# Cell 8: Pre-filter sources based on quality criteria
# Remove sites that are too old, too short, or match blocklist
from urllib.parse import urlparse

def should_keep_source(source: dict) -> tuple[bool, str]:
    """Determine if source should be kept based on quality criteria."""
    url = source["url"].lower()
    
    # Check blocklist again (post-scrape)
    for pattern in BLOCKLIST["url_patterns"]:
        if pattern in url:
            return False, f"BLOCKLIST pattern: {pattern}"
    
    hostname = urlparse(url).netloc.lower()
    for blocked in BLOCKLIST["domains"]:
        if blocked in hostname:
            return False, f"BLOCKLIST domain: {blocked}"
    
    # Minimum content length
    if len(source["text"]) < 300:
        return False, f"Too short ({len(source['text'])} chars)"
    
    # Currency check (already done during scraping, but double-check)
    age = source_age_months(source.get("date"))
    if age is not None and age > MAX_SOURCE_AGE_MONTHS:
        return False, f"Too old ({age} months)"
    
    return True, "OK"

# Apply filters
filtered_sources = []
filter_reasons = []

print("Pre-filtering sources...")
print("=" * 60)

for s in scraped_sources:
    keep, reason = should_keep_source(s)
    if keep:
        filtered_sources.append(s)
    else:
        filter_reasons.append({
            "title": s.get("title", "?"),
            "url": s["url"],
            "reason": reason
        })

print(f"Kept {len(filtered_sources)} sources")
print(f"Filtered out {len(filter_reasons)} sources")

if filter_reasons:
    print(f"\nFiltered sources (first 10):")
    for f in filter_reasons[:10]:
        print(f"  ✗ {f['reason']}: {f['title'][:60]}")
        print(f"    {f['url']}")

print(f"\nFiltered sources by tier:")
tier_filtered = {}
for s in filtered_sources:
    tier = s.get("search_pass", "Unknown")
    tier_filtered[tier] = tier_filtered.get(tier, 0) + 1

for tier, count in tier_filtered.items():
    pct = count / len(filtered_sources) * 100 if filtered_sources else 0
    print(f"  {tier}: {count} sources ({pct:.1f}%)")

Kept 14 sources, dropped 14

Dropped:
  ✗ too old (56 months, threshold 24): Account-Based Everything: What Happens When Intent Signals B
    https://www.forrester.com/report/account-based-everything-what-happens-when-intent-signals-become-ubiquitous-in-b2b-marketing/RES176089
  ✗ too old (96 months, threshold 24): Account-Based Marketing (ABM): The Ultimate SiriusDecisions 
    https://www.forrester.com/blogs/what-is-account-based-marketing/
  ✗ too old (63 months, threshold 24): Are You Ready For The Convergence Of ABM And Demand Technolo
    https://www.forrester.com/blogs/are-you-ready-for-the-convergence-of-abm-and-demand-technologies/
  ✗ too old (57 months, threshold 24): Consider Content Engagement Solutions To Enhance Digital Buy
    https://www.forrester.com/blogs/consider-content-engagement-solutions-to-enhance-digital-buyer-interactions/
  ✗ too old (66 months, threshold 24): Digitized Journeys, Expanding Options, And Maturing Segments
    https://www.forrester.com/blogs/di

In [ ]:
# Cell 9: Score sources using LLM based on slot-filling criteria
# Evaluates how well each source fills the 5 required slots
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
import json, time as _time

LLM_DELAY = 0.2  # Rate limiting for LLM calls

class SourceScore(BaseModel):
    """Structured output for source quality scoring."""
    slot_definition: bool = Field(description="Contains a category definition")
    slot_capabilities: bool = Field(description="Lists core capabilities of the software")
    slot_boundaries: bool = Field(description="Distinguishes from adjacent/related categories")
    slot_buyer_use: bool = Field(description="Describes buyer persona or use case")
    slot_vendors: bool = Field(description="Names representative vendors/products")
    slots_filled: int = Field(description="Count of slots filled (0-5)")
    uses_function_verbs: bool = Field(description="Uses expert verbs like orchestrate, unify, score, route, match, segment, personalize")
    vendor_count: int = Field(description="Number of distinct vendors/products mentioned")
    vendor_names: list[str] = Field(description="List of distinct vendor/product names found")
    is_sme_content: bool = Field(description="Appears to be written by or for subject-matter experts, not SEO/marketing fluff")
    byline_quality: str = Field(description="'named_analyst' if byline is a recognized analyst, 'named_author' if any named author, 'no_byline' if anonymous")
    single_vendor_bias: bool = Field(description="True if source primarily promotes a single vendor rather than providing neutral analysis")
    relevance_score: int = Field(description="1-10 overall relevance to defining the software category")
    reasoning: str = Field(description="Brief explanation of the score")

SCORING_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are evaluating a web source for its usefulness in defining the software category "{category}".

Score it on these criteria:

1. SLOT-FILL: Does it contain (a) a definition, (b) core capabilities, (c) boundaries vs adjacent categories, (d) buyer/use case, (e) representative vendors? Count how many of these 5 slots it fills.

2. FUNCTION-VERBS: Does it use expert verbs (orchestrate, unify, score, route, segment, personalize, align, prioritize, automate, enrich, match) rather than SEO adjectives (better, smarter, faster, top, best)?

3. BYLINE QUALITY: 
   - "named_analyst" = recognized analyst at a known firm (Gartner, Forrester, Constellation, ISG, etc.)
   - "named_author" = has a named author but not clearly a recognized analyst
   - "no_byline" = anonymous, staff-writer, or no author attribution

4. VENDOR DIVERSITY & BIAS: How many distinct vendors are named? List them. A source naming 5+ vendors from different corporate families shows range. Flag single_vendor_bias if the source primarily promotes one vendor.

5. SME CONTENT: Is this analyst/expert content or marketing fluff? Consider the depth of analysis, presence of frameworks, and whether it reads as editorial work vs promotional material.

6. RELEVANCE: How useful is this source for writing a definitive category page (1-10)?
   - 8-10: Fills 4-5 slots, uses function-verbs, SME-authored, multi-vendor
   - 5-7: Fills 2-3 slots or has partial quality signals
   - 1-4: Off-topic, thin, or promotional

Return valid JSON matching the schema."""),
    ("human", """Source URL: {url}
Title: {title}
Author: {author}
Date: {date}
Host: {hostname}
Source age: {source_age}

Content (first 3000 chars):
{text}"""),
])

# Initialize LLM
llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0, api_key=OPENAI_API_KEY)
structured_llm = llm.with_structured_output(SourceScore)
chain = SCORING_PROMPT | structured_llm

# Score all filtered sources
scored_sources = []
cache_hits = 0

print(f"Scoring {len(filtered_sources)} sources with LLM...")
print("=" * 60)

for i, s in enumerate(filtered_sources):
    print(f"[{i+1}/{len(filtered_sources)}] Scoring: {s['title'][:60]}…", end=" ")
    
    # Prepare input
    text_trunc = s["text"][:3000]
    age = source_age_months(s.get("date"))
    age_str = f"{age} months" if age is not None else "unknown"
    
    invoke_params = {
        "category": TEST_CATEGORY,
        "url": s["url"],
        "title": s["title"] or "Unknown",
        "author": s["author"] or "Unknown",
        "date": s["date"] or "Unknown",
        "hostname": s["hostname"] or "Unknown",
        "source_age": age_str,
        "text": text_trunc,
    }
    
    # Check cache
    cache_key_parts = ("scoring", TEST_CATEGORY, s["url"], s["title"] or "Unknown",
                       s["author"] or "Unknown", s["date"] or "Unknown",
                       s["hostname"] or "Unknown", text_trunc)
    cached = cache_get("llm_scoring", *cache_key_parts)
    
    if cached is not None:
        score = SourceScore(**cached)
        cache_hits += 1
        scored_sources.append({"source": s, "score": score})
        print(f"✓ relevance={score.relevance_score}/10, slots={score.slots_filled}/5 (cached)")
        continue
    
    try:
        score = chain.invoke(invoke_params)
        # Cache result
        cache_set("llm_scoring", score.model_dump(), *cache_key_parts)
        scored_sources.append({"source": s, "score": score})
        print(f"✓ relevance={score.relevance_score}/10, slots={score.slots_filled}/5")
        _time.sleep(LLM_DELAY)
    except Exception as e:
        print(f"✗ {e}")

# Sort by relevance score
scored_sources.sort(key=lambda x: x["score"].relevance_score, reverse=True)

print(f"\n{'=' * 60}")
print(f"SCORING COMPLETE:")
print(f"  Sources scored: {len(scored_sources)}")
print(f"  Cache hits: {cache_hits}")
print(f"  Avg relevance: {sum(item['score'].relevance_score for item in scored_sources) / len(scored_sources):.1f}/10")

cache_stats()

c:\Users\Abhishek A\Defining_Category\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SCORING PROMPT TEMPLATE:

── You are evaluating a… (SystemMessagePromptTemplate) ──

── Source URL: {url}
Ti… (HumanMessagePromptTemplate) ──

Input variables: ['author', 'category', 'date', 'hostname', 'source_age', 'text', 'title', 'url']
[1/16] Scoring: Jessie Johnson… ✓ relevance=2/10, slots=0/5
[2/16] Scoring: Terry Flaherty… ✓ relevance=2/10, slots=0/5
[3/16] Scoring: ISG Software Research Analyst Perspectives | intelligent mar… ✓ relevance=2/10, slots=0/5
[4/16] Scoring: Account-Based Marketing (ABM) Archives… ✓ relevance=4/10, slots=1/5
[5/16] Scoring: Distributing Responsibilities Between An Account-Based Marke… ✓ relevance=8/10, slots=4/5
[6/16] Scoring: Establishing An Account-Based Marketing Charter | Forrester… ✓ relevance=8/10, slots=4/5
[7/16] Scoring: How ABM advertising accelerates B2B growth… ✓ relevance=9/10, slots=5/5
[8/16] Scoring: Accelerate Account-Based Marketing: Orient Your Strategy wit… ✓ relevance=8/10, slots=4/5
[9/16] Scoring: Accelerate Account-Based Mar

In [ ]:
# Cell 10: Display scored sources and apply further filtering
# Show top sources and filter based on score thresholds
print(f"Top {min(10, len(scored_sources))} scored sources:\n")
for i, item in enumerate(scored_sources[:10]):
    s = item["source"]
    sc = item["score"]
    print(f"#{i+1}  Relevance: {sc.relevance_score}/10 | Slots: {sc.slots_filled}/5 | Vendors: {sc.vendor_count}")
    print(f"  Title : {s['title']}")
    print(f"  Author: {s['author'] or 'n/a'} | Date: {s['date'] or 'n/a'} | Host: {s['hostname'] or 'n/a'}")
    print(f"  URL   : {s['url']}")
    print(f"  Byline: {sc.byline_quality} | Expert verbs: {sc.uses_function_verbs} | SME: {sc.is_sme_content}")
    print(f"  Slots → Def:{sc.slot_definition} Cap:{sc.slot_capabilities} Bound:{sc.slot_boundaries} Buyer:{sc.slot_buyer_use} Vendors:{sc.slot_vendors}")
    if sc.vendor_names:
        print(f"  Vendors named: {', '.join(sc.vendor_names)}")
    print(f"  Reasoning: {sc.reasoning}")
    print()

# Apply further filtering based on scores
print("=" * 70)
print("APPLYING SCORE-BASED FILTERING")
print("=" * 70)

# Filter criteria
MIN_RELEVANCE = 5  # Minimum relevance score
MIN_SLOTS = 2      # Minimum slots filled
MAX_SINGLE_VENDOR_BIAS = True  # Exclude single-vendor-biased sources

high_quality_sources = []
filtered_out = []

for item in scored_sources:
    s = item["source"]
    sc = item["score"]
    
    # Apply filters
    if sc.relevance_score < MIN_RELEVANCE:
        filtered_out.append((s, f"Low relevance ({sc.relevance_score}/10)"))
    elif sc.slots_filled < MIN_SLOTS:
        filtered_out.append((s, f"Too few slots ({sc.slots_filled}/5)"))
    elif sc.single_vendor_bias and MAX_SINGLE_VENDOR_BIAS:
        filtered_out.append((s, "Single vendor bias"))
    else:
        high_quality_sources.append(item)

print(f"High-quality sources: {len(high_quality_sources)}")
print(f"Filtered out: {len(filtered_out)}")

if filtered_out:
    print(f"\nFiltered out sources (first 10):")
    for s, reason in filtered_out[:10]:
        print(f"  ✗ {reason}: {s['title'][:60]}")

# Quality breakdown of final sources
print(f"\nFinal quality breakdown:")
if high_quality_sources:
    avg_relevance = sum(item["score"].relevance_score for item in high_quality_sources) / len(high_quality_sources)
    avg_slots = sum(item["score"].slots_filled for item in high_quality_sources) / len(high_quality_sources)
    named_analysts = sum(1 for item in high_quality_sources if item["score"].byline_quality == "named_analyst")
    
    print(f"  Sources: {len(high_quality_sources)}")
    print(f"  Avg relevance: {avg_relevance:.1f}/10")
    print(f"  Avg slots filled: {avg_slots:.1f}/5")
    print(f"  Named analysts: {named_analysts} ({named_analysts/len(high_quality_sources)*100:.1f}%)")
    
    # Tier breakdown
    tier_final = {}
    for item in high_quality_sources:
        tier = item["source"].get("search_pass", "Unknown")
        tier_final[tier] = tier_final.get(tier, 0) + 1
    
    print(f"\nFinal sources by tier:")
    for tier, count in tier_final.items():
        pct = count / len(high_quality_sources) * 100
        print(f"  {tier}: {count} sources ({pct:.1f}%)")
else:
    print(f"  No sources passed filtering!")

# Use high_quality_sources for synthesis
top_sources = high_quality_sources

#1  Relevance: 10/10 | Slots: 5/5 | Vendors: 12
  Title : Converging Platforms For Greater Efficiency: The Rise Of Revenue Marketing Platforms
  Author: Kelvin Gee | Date: 2024-07-25 | Host: Forrester
  URL   : https://www.forrester.com/blogs/converging-platforms-for-greater-efficiency-the-rise-of-revenue-marketing-platforms/
  Byline: named_analyst | Expert verbs: True | SME: True | Bias: False
  Slots → Def:True Cap:True Bound:True Buyer:True Vendors:True
  Vendors named: Vendor 1, Vendor 2, Vendor 3, Vendor 4, Vendor 5, Vendor 6, Vendor 7, Vendor 8, Vendor 9, Vendor 10, Vendor 11, Vendor 12
  Reasoning: The source provides a comprehensive definition of revenue marketing platforms, lists core capabilities, distinguishes it from adjacent categories like MAPs, describes buyer use cases, and names multiple vendors. It uses expert verbs throughout and is authored by a recognized analyst at Forrester, indicating high-quality SME content. Overall, it is highly relevant for defining the sof

In [ ]:
# Cell 11: LLM Synthesis - Generate the category page
# All passed websites go to LLM synthesizer to generate the category page
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
import json, time as _time

class CategoryPage(BaseModel):
    """Structured output for category page synthesis."""
    category_name: str = Field(description="Primary category name")
    aliases: list[str] = Field(description="Known aliases for this category")
    definition: str = Field(description="2-4 sentence category definition synthesized from multiple sources")
    core_capabilities: list[str] = Field(description="Core software capabilities (use function-verbs)")
    boundaries: str = Field(description="What this category is NOT; how it differs from adjacent categories")
    buyer_use_case: str = Field(description="Who buys this software and why")
    representative_vendors: list[str] = Field(description="Named vendors from across multiple sources")
    category_drift: str = Field(description="Where analyst firms disagree on scope, naming, or existence")
    source_count: int = Field(description="Number of sources used in synthesis")
    confidence: str = Field(description="high/medium/low based on source coverage and consensus")

SYNTHESIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are writing a category definition page. The page must be written in editorial voice — no direct quotation from sources.

RULES:
- Fill five slots in order: definition, core capabilities, boundaries, buyer/use case, representative vendors
- DEFINITION: 2-4 sentences describing what the SOFTWARE does, not the methodology
- CORE CAPABILITIES: Use function-verbs (orchestrate, unify, score, route, segment, personalize, align, prioritize, automate, enrich, match)
- BOUNDARIES: Name specific adjacent categories this is NOT
- BUYER/USE CASE: Name specific buyer roles and organization types
- VENDORS: Must come from across multiple sources, not a single source
- Address category drift if firms disagree on scope or naming

Category: {category}
Maturity: {maturity}
Aliases: {aliases}"""),
    ("human", """Here are the top-scoring sources to synthesize from:

{sources_text}

Synthesize these into a single category page. Return structured JSON."""),
])

# Prepare sources for synthesis
if not top_sources:
    print("❌ ERROR: No sources available for synthesis!")
    print("Check earlier cells for filtering issues.")
else:
    print(f"Synthesizing category page from {len(top_sources)} high-quality sources...")
    print("=" * 70)
    
    # Prepare sources block
    sources_block = ""
    for i, item in enumerate(top_sources):
        s = item["source"]
        sc = item["score"]
        sources_block += f"\n--- SOURCE {i+1} (relevance {sc.relevance_score}/10, slots {sc.slots_filled}/5) ---\n"
        sources_block += f"Title: {s['title']}\n"
        sources_block += f"Author: {s['author'] or 'Unknown'} | Host: {s['hostname'] or 'Unknown'} | Date: {s['date'] or 'Unknown'}\n"
        sources_block += f"Byline quality: {sc.byline_quality} | Vendors named: {', '.join(sc.vendor_names) if sc.vendor_names else 'none'}\n"
        sources_block += f"Content:\n{s['text'][:4000]}\n"
    
    # Check synthesis cache
    synth_cache_key = ("synthesis", TEST_CATEGORY, ", ".join(CATEGORY_ALIASES), sources_block)
    cached_synth = cache_get("llm_synthesis", *synth_cache_key)
    
    if cached_synth is not None:
        category_page = CategoryPage(**cached_synth)
        print("✓ Synthesis loaded from cache!")
    else:
        # Generate synthesis
        synth_llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0.2, api_key=OPENAI_API_KEY)
        synth_chain = SYNTHESIS_PROMPT | synth_llm.with_structured_output(CategoryPage)
        
        try:
            category_page = synth_chain.invoke({
                "category": TEST_CATEGORY,
                "maturity": CATEGORY_MATURITY,
                "aliases": ", ".join(CATEGORY_ALIASES),
                "sources_text": sources_block,
            })
            
            # Cache the result
            cache_set("llm_synthesis", category_page.model_dump(), *synth_cache_key)
            print("✓ Synthesis complete (cached for next run)!")
            
        except Exception as e:
            print(f"❌ Synthesis failed: {e}")
            category_page = None
    
    # Display results if synthesis succeeded
    if category_page:
        print(f"\n{'=' * 70}")
        print(f"  CATEGORY PAGE: {category_page.category_name}")
        print(f"{'=' * 70}")
        print(f"\nAliases: {', '.join(category_page.aliases)}")
        print(f"Confidence: {category_page.confidence} | Sources used: {category_page.source_count}")
        
        print(f"\n{'─' * 70}")
        print("1. DEFINITION")
        print(f"{'─' * 70}")
        print(category_page.definition)
        
        print(f"\n{'─' * 70}")
        print("2. CORE CAPABILITIES")
        print(f"{'─' * 70}")
        for cap in category_page.core_capabilities:
            print(f"  • {cap}")
        
        print(f"\n{'─' * 70}")
        print("3. BOUNDARIES")
        print(f"{'─' * 70}")
        print(category_page.boundaries)
        
        print(f"\n{'─' * 70}")
        print("4. BUYER / USE CASE")
        print(f"{'─' * 70}")
        print(category_page.buyer_use_case)
        
        print(f"\n{'─' * 70}")
        print("5. REPRESENTATIVE VENDORS")
        print(f"{'─' * 70}")
        for v in category_page.representative_vendors:
            print(f"  • {v}")
        
        if category_page.category_drift:
            print(f"\n{'─' * 70}")
            print("6. CATEGORY DRIFT / ANALYST DISAGREEMENT")
            print(f"{'─' * 70}")
            print(category_page.category_drift)
        
        print(f"\n{'=' * 70}")
        print("Sources used in synthesis:")
        for i, item in enumerate(top_sources):
            s = item["source"]
            sc = item["score"]
            print(f"  [{i+1}] {s['title']} — {s['author'] or 'n/a'} ({s['hostname'] or 'n/a'})")
            print(f"      Relevance: {sc.relevance_score}/10, Byline: {sc.byline_quality}, Vendors: {len(sc.vendor_names)}")

cache_stats()

Synthesizing from 7 sources (23063 chars total)…

✓ Synthesis complete (cached for next run)!
  llm_scoring: 16 entries
  llm_synthesis: 1 entries
  serper: 8 entries


In [ ]:
# Cell 12: Export results and final summary
# Save category page and all intermediate results
import json, pathlib
from datetime import datetime

# Create output directory
output_dir = pathlib.Path("output")
output_dir.mkdir(exist_ok=True)

# Export category page
if 'category_page' in locals() and category_page:
    cat_slug = TEST_CATEGORY.lower().replace(" ", "_").replace("-", "_")
    page_path = output_dir / f"{cat_slug}_page.json"
    page_path.write_text(json.dumps(category_page.model_dump(), indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"✓ Category page exported to: {page_path}")

# Export all search results
all_links_export = []
for i, r in enumerate(all_results):
    all_links_export.append({
        "index": i + 1,
        "search_pass": r["search_pass"],
        "title": r["title"],
        "url": r["url"],
        "snippet": r.get("snippet", ""),
        "query_alias": r.get("query_alias", ""),
    })

links_path = output_dir / f"{cat_slug}_all_searched_links.json"
links_path.write_text(json.dumps(all_links_export, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"✓ All searched links exported to: {links_path}")

# Export scraped sources
if 'scraped_sources' in locals():
    scraped_export = []
    for s in scraped_sources:
        scraped_export.append({
            "url": s["url"],
            "title": s["title"],
            "author": s["author"],
            "date": s["date"],
            "hostname": s["hostname"],
            "search_pass": s.get("search_pass", ""),
            "text_length": len(s["text"]),
        })
    
    scraped_path = output_dir / f"{cat_slug}_scraped_sources.json"
    scraped_path.write_text(json.dumps(scraped_export, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"✓ Scraped sources exported to: {scraped_path}")

# Export scored sources
if 'scored_sources' in locals():
    scores_export = []
    for item in scored_sources:
        s = item["source"]
        sc = item["score"]
        scores_export.append({
            "url": s["url"],
            "title": s["title"],
            "author": s["author"],
            "date": s["date"],
            "hostname": s["hostname"],
            "search_pass": s.get("search_pass", ""),
            "text_length": len(s["text"]),
            "score": sc.model_dump(),
        })
    
    scores_path = output_dir / f"{cat_slug}_scored_sources.json"
    scores_path.write_text(json.dumps(scores_export, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"✓ Scored sources exported to: {scores_path}")

# Export final high-quality sources
if 'top_sources' in locals():
    final_export = []
    for item in top_sources:
        s = item["source"]
        sc = item["score"]
        final_export.append({
            "url": s["url"],
            "title": s["title"],
            "author": s["author"],
            "date": s["date"],
            "hostname": s["hostname"],
            "search_pass": s.get("search_pass", ""),
            "relevance_score": sc.relevance_score,
            "slots_filled": sc.slots_filled,
            "byline_quality": sc.byline_quality,
            "vendor_names": sc.vendor_names,
        })
    
    final_path = output_dir / f"{cat_slug}_final_sources.json"
    final_path.write_text(json.dumps(final_export, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"✓ Final high-quality sources exported to: {final_path}")

# Create final summary
print(f"\n{'=' * 70}")
print(f"PIPELINE COMPLETE - {TEST_CATEGORY}")
print(f"{'=' * 70}")

summary = {
    "category": TEST_CATEGORY,
    "timestamp": datetime.now().isoformat(),
    "pipeline_stats": {
        "total_search_queries": total_queries if 'total_queries' in locals() else 0,
        "urls_found": len(all_results) if 'all_results' in locals() else 0,
        "urls_scraped": len(scraped_sources) if 'scraped_sources' in locals() else 0,
        "sources_filtered": len(filtered_sources) if 'filtered_sources' in locals() else 0,
        "sources_scored": len(scored_sources) if 'scored_sources' in locals() else 0,
        "sources_final": len(top_sources) if 'top_sources' in locals() else 0,
    },
    "quality_metrics": {
        "scrape_success_rate": len(scraped_sources)/len(all_results)*100 if 'scraped_sources' in locals() and 'all_results' in locals() else 0,
        "avg_relevance": sum(item["score"].relevance_score for item in scored_sources)/len(scored_sources) if 'scored_sources' in locals() else 0,
        "synthesis_confidence": category_page.confidence if 'category_page' in locals() and category_page else "unknown",
    }
}

print(f"Pipeline Statistics:")
for key, value in summary["pipeline_stats"].items():
    print(f"  {key}: {value}")

print(f"\nQuality Metrics:")
for key, value in summary["quality_metrics"].items():
    if isinstance(value, float):
        print(f"  {key}: {value:.1f}")
    else:
        print(f"  {key}: {value}")

# Save summary
summary_path = output_dir / f"{cat_slug}_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\n✓ Summary exported to: {summary_path}")

print(f"\n🎉 All exports complete in {output_dir} directory!")

  CATEGORY PAGE: Account-Based Marketing

Aliases: Account-Based Marketing, ABM, Account-Based Marketing Platforms, ABM platforms, Account-Based Everything, ABX, Account-Based Experience
Confidence: high | Sources used: 7

──────────────────────────────────────────────────────────────────────
1. DEFINITION
──────────────────────────────────────────────────────────────────────
Account-Based Marketing (ABM) is a strategic approach in B2B marketing that focuses on targeting specific accounts or buying groups rather than individual leads. This software category enables organizations to design and execute highly personalized marketing campaigns tailored to the unique needs and characteristics of identified accounts, facilitating deeper engagement and alignment between sales and marketing efforts.

──────────────────────────────────────────────────────────────────────
2. CORE CAPABILITIES
──────────────────────────────────────────────────────────────────────
  • orchestrate
  • personalize
 

In [ ]:
# Cell 12: Export ALL searched links for Phase 1 task
import json, pathlib

# Create export directory
output_dir = pathlib.Path("output")
output_dir.mkdir(exist_ok=True)

# Export all searched links with metadata
all_links_export = []
for i, r in enumerate(all_results):
    all_links_export.append({
        "index": i + 1,
        "search_pass": r["search_pass"],
        "title": r["title"],
        "url": r["url"],
        "snippet": r.get("snippet", ""),
        "query_alias": r.get("query_alias", ""),
    })

# Save to JSON
links_path = output_dir / f"{TEST_CATEGORY.lower().replace(' ', '_')}_all_searched_links.json"
links_path.write_text(json.dumps(all_links_export, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"\n{'=' * 70}")
print(f"PHASE 1 EXPORT COMPLETE")
print(f"{'=' * 70}")
print(f"✓ Exported {len(all_links_export)} searched links to:")
print(f"  {links_path}")
print(f"\nFile includes: index, search_pass, title, url, snippet, query_alias")
print(f"Ready for Phase 1 review and analysis.")

SECOND-PASS GAP ANALYSIS
  ⚠️  Sources from only 2 distinct hosts — low diversity

1 gaps identified — consider targeted follow-up searches.

Source host diversity: forrester, idc


In [ ]:
# Cell 14: Ground truth benchmark for ABM category
# Known high-quality sources that should appear in search results
from urllib.parse import urlparse

ABM_GROUND_TRUTH = {
    # MUST-HAVE: Core analyst firms that definitely cover ABM
    "must_have": {
        "gartner.com": {
            "reason": "Gartner has dedicated ABM/ABX research and frameworks",
            "expected_content": ["Account-Based Everything framework", "ABM Magic Quadrant", "ABM vendor guides"],
            "priority": "critical"
        },
        "forrester.com": {
            "reason": "Forrester has dedicated ABM analysts (John Arnold, Jessie Johnson)",
            "expected_content": ["ABM Wave reports", "ABM vendor evaluations", "ABM best practices"],
            "priority": "critical"
        },
        "idc.com": {
            "reason": "IDC covers ABM in their marketing technology research",
            "expected_content": ["ABM market sizing", "ABM vendor assessments"],
            "priority": "critical"
        }
    },
    
    # GOOD: Independent analysts with strong ABM coverage
    "good": {
        "constellationr.com": {
            "reason": "Constellation Research covers ABM under customer experience",
            "expected_content": ["ABM research notes", "vendor connections"],
            "priority": "high"
        },
        "isg-one.com": {
            "reason": "ISG/Ventana Research has dedicated ABM research (Keith Dawson)",
            "expected_content": ["ABM vendor comparisons", "ABM implementation guides"],
            "priority": "high"
        },
        "aragonresearch.com": {
            "reason": "Aragon covers ABM in their B2B marketing research",
            "expected_content": ["ABM technology reports", "vendor evaluations"],
            "priority": "high"
        },
        "gigaom.com": {
            "reason": "GigaOm covers marketing technology including ABM",
            "expected_content": ["ABM market analysis", "technology trends"],
            "priority": "medium"
        }
    },
    
    # ACCEPTABLE: Trade publications with good ABM coverage
    "acceptable": {
        "chiefmartec.com": {
            "reason": "Scott Brinker's martech blog includes ABM coverage",
            "expected_content": ["ABM landscape charts", "vendor ecosystem"],
            "priority": "medium"
        },
        "martech.org": {
            "reason": "MarTech Tribe covers ABM tools and strategies",
            "expected_content": ["ABM tool reviews", "implementation tips"],
            "priority": "medium"
        },
        "adexchanger.com": {
            "reason": "AdExchanger covers B2B marketing including ABM",
            "expected_content": ["ABM case studies", "vendor news"],
            "priority": "low"
        }
    },
    
    # SHOULD NOT HAVE: Vendor sites that should be filtered out
    "should_not_have": {
        "optimizely.com": "Personalization vendor, not analyst research",
        "salesforce.com": "CRM vendor with marketing content",
        "demandbase.com": "ABM vendor with promotional content",
        "mutinyhq.com": "ABM vendor blog",
        "clay.com": "ABM vendor content",
        "strategicabm.com": "ABM agency/vendor",
        "factors.ai": "ABM vendor",
        "cognism.com": "B2B data vendor",
        "linkedin.com": "Social platform, not analyst research",
        "wikipedia.org": "General knowledge, not expert analysis"
    }
}

def evaluate_ground_truth_coverage(results):
    """Evaluate search results against ground truth benchmark."""
    
    # Extract domains from results
    result_domains = [urlparse(r["url"]).netloc.lower() for r in results]
    
    coverage = {
        "must_have_found": [],
        "must_have_missing": [],
        "good_found": [],
        "good_missing": [],
        "acceptable_found": [],
        "acceptable_missing": [],
        "should_not_have_found": [],
        "should_not_have_missing": [],
        "coverage_scores": {}
    }
    
    # Check must-have sources
    for domain, info in ABM_GROUND_TRUTH["must_have"].items():
        found = any(domain in rd for rd in result_domains)
        if found:
            coverage["must_have_found"].append((domain, info["priority"]))
        else:
            coverage["must_have_missing"].append((domain, info["priority"]))
    
    # Check good sources
    for domain, info in ABM_GROUND_TRUTH["good"].items():
        found = any(domain in rd for rd in result_domains)
        if found:
            coverage["good_found"].append((domain, info["priority"]))
        else:
            coverage["good_missing"].append((domain, info["priority"]))
    
    # Check acceptable sources
    for domain, info in ABM_GROUND_TRUTH["acceptable"].items():
        found = any(domain in rd for rd in result_domains)
        if found:
            coverage["acceptable_found"].append((domain, info["priority"]))
        else:
            coverage["acceptable_missing"].append((domain, info["priority"]))
    
    # Check for unwanted sources
    for domain, reason in ABM_GROUND_TRUTH["should_not_have"].items():
        found = any(domain in rd for rd in result_domains)
        if found:
            coverage["should_not_have_found"].append((domain, reason))
        else:
            coverage["should_not_have_missing"].append((domain, reason))
    
    # Calculate coverage scores
    total_must_have = len(ABM_GROUND_TRUTH["must_have"])
    total_good = len(ABM_GROUND_TRUTH["good"])
    total_acceptable = len(ABM_GROUND_TRUTH["acceptable"])
    
    coverage["coverage_scores"] = {
        "must_have_coverage": len(coverage["must_have_found"]) / total_must_have * 100,
        "good_coverage": len(coverage["good_found"]) / total_good * 100,
        "acceptable_coverage": len(coverage["acceptable_found"]) / total_acceptable * 100,
        "unwanted_sources": len(coverage["should_not_have_found"]),
        "overall_good_source_coverage": (
            len(coverage["must_have_found"]) + 
            len(coverage["good_found"]) + 
            len(coverage["acceptable_found"])
        ) / (total_must_have + total_good + total_acceptable) * 100
    }
    
    return coverage

def print_ground_truth_evaluation(coverage):
    """Print detailed ground truth evaluation."""
    
    print("=" * 70)
    print("GROUND TRUTH COVERAGE EVALUATION")
    print("=" * 70)
    
    scores = coverage["coverage_scores"]
    print(f"\n📊 COVERAGE SCORES:")
    print(f"  Must-have sources: {scores['must_have_coverage']:.1f}% ({len(coverage['must_have_found'])}/{len(ABM_GROUND_TRUTH['must_have'])})")
    print(f"  Good sources: {scores['good_coverage']:.1f}% ({len(coverage['good_found'])}/{len(ABM_GROUND_TRUTH['good'])})")
    print(f"  Acceptable sources: {scores['acceptable_coverage']:.1f}% ({len(coverage['acceptable_found'])}/{len(ABM_GROUND_TRUTH['acceptable'])})")
    print(f"  Overall good coverage: {scores['overall_good_source_coverage']:.1f}%")
    print(f"  Unwanted sources found: {scores['unwanted_sources']}")
    
    # Critical: Must-have sources
    if coverage["must_have_missing"]:
        print(f"\n🚨 CRITICAL - MISSING MUST-HAVE SOURCES:")
        for domain, priority in coverage["must_have_missing"]:
            print(f"  ❌ {domain} ({priority}) - {ABM_GROUND_TRUTH['must_have'][domain]['reason']}")
    else:
        print(f"\n✅ ALL CRITICAL MUST-HAVE SOURCES FOUND!")
    
    # Good sources missing
    if coverage["good_missing"]:
        print(f"\n⚠️  MISSING GOOD SOURCES:")
        for domain, priority in coverage["good_missing"]:
            print(f"  ⚠️  {domain} ({priority}) - {ABM_GROUND_TRUTH['good'][domain]['reason']}")
    
    # Unwanted sources found
    if coverage["should_not_have_found"]:
        print(f"\n❌ UNWANTED SOURCES FOUND (should be filtered out):")
        for domain, reason in coverage["should_not_have_found"]:
            print(f"  ❌ {domain} - {reason}")
    else:
        print(f"\n✅ NO UNWANTED SOURCES FOUND!")
    
    # Summary
    print(f"\n📋 SUMMARY:")
    must_have_ok = len(coverage["must_have_missing"]) == 0
    unwanted_ok = len(coverage["should_not_have_found"]) == 0
    overall_good = scores["overall_good_source_coverage"] >= 70
    
    if must_have_ok and unwanted_ok and overall_good:
        print(f"  ✅ SEARCH QUALITY: GOOD - All critical sources found, no unwanted sources")
    elif must_have_ok and unwanted_ok:
        print(f"  ⚠️  SEARCH QUALITY: ACCEPTABLE - Critical sources OK, but could find more good sources")
    elif not must_have_ok:
        print(f"  ❌ SEARCH QUALITY: POOR - Missing critical analyst sources")
    elif not unwanted_ok:
        print(f"  ❌ SEARCH QUALITY: POOR - Vendor sites not properly filtered")
    
    return scores

# Run ground truth evaluation
coverage_results = evaluate_ground_truth_coverage(all_results)
ground_truth_scores = print_ground_truth_evaluation(coverage_results)